<a href="https://colab.research.google.com/github/RajendarKumar/langgraph/blob/main/refund_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "groq>=0.4.0"
# Install pydantic for schema validation
!pip install --quiet pydantic


In [38]:
from google.colab import userdata
key = userdata.get('GROQ_API_KEY')

In [43]:
from typing import Optional, Literal
from pydantic import BaseModel, Field, ValidationError

# Define Pydantic models for output validation
class EmailDetails(BaseModel):
    category: Literal["refund", "shipping", "complaint", "other"]
    order_number: Optional[str]
    amount: float # Assuming amount can be a float, based on 'number' type
    order_date: Optional[str] = Field(pattern="^\\d{4}-\\d{2}-\\d{2}$") # YYYY-MM-DD format

class RefundPolicyResult(BaseModel):
    refund_possible: Literal["YES", "NO", "NOT SURE"]
    reason: str

In [44]:
from groq import Groq
import json
import datetime
import re
import pandas as pd


client = Groq(api_key=key)

def refund_rules_list(refund_policy:str):
  refund_rules_list = []
  # Split the string by rule number (e.g., '1. ' or '2. ')
  rules = re.split(r'\d+\.\s*', refund_policy)

  # Filter out empty strings that might result from the split and process each rule
  for i, rule_text in enumerate(rules):
      if rule_text.strip(): # Ensure the rule text is not empty
          rule_obj = {
              "rule": f"Rule {i}", # Assign a generic rule name, or infer from text if possible
              "value": rule_text.strip()
          }
          refund_rules_list.append(rule_obj)
  print(json.dumps(refund_rules_list, indent=2))
  return refund_rules_list
def extract_email_details(email_body: str, model: str = "openai/gpt-oss-120b"):
    """Extracts structured details from an email body using an LLM and validates against schema."""
    user_prompt = f"""
    Extract the following details from the email body into a JSON object:
    - category (refund, shipping, complaint, other)
    - order_number (string or null)
    - amount (number)
    - order_date (string, format YYYY-MM-DD)

    Email body: "{email_body}"

    JSON Schema:
    ```json
    {{
      "category": {{ "type": "string", "enum": ["refund", "shipping", "complaint", "other"] }},
      "order_number": {{ "type": ["string", "null"] }},
      "amount": {{ "type": "number" }},
      "order_date": {{ "type": ["string", "null"], "format": "YYYY-MM-DD" }}
    }}
    ```
    Ensure the output is valid JSON according to this schema.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0.0, # Make it deterministic for data extraction
        max_tokens=1024,
        top_p=1.0,
        stream=False,
        response_format={ "type": "json_object" }
    )
    print(f'email evaluation cost : {response.usage}')

    parsed_content = json.loads(response.choices[0].message.content)

    try:
        # Validate the parsed content against the Pydantic model
        validated_data = EmailDetails.model_validate(parsed_content)
        return validated_data.model_dump() # Return as dict
    except ValidationError as e:
        print(f"[ERROR] Email details validation failed: {e}")
        # In a real application, you might log this error and handle it gracefully
        # For now, we'll return the raw parsed content, but a robust system might raise it
        return parsed_content


def load_orders(file_name: str, file_ext: str):
    """Loads orders from a file (CSV or JSON) and returns them as a list of dictionaries."""
    orders_df = None
    if file_ext.lower() == '.csv':
        try:
            orders_df = pd.read_csv(file_name)
        except FileNotFoundError:
            print(f"Error: CSV file not found at {file_name}")
            return []
    elif file_ext.lower() == '.json':
        try:
            orders_df = pd.read_json(file_name)
        except FileNotFoundError:
            print(f"Error: JSON file not found at {file_name}")
            return []
    else:
        print(f"Error: Unsupported file extension: {file_ext}. Only .csv and .json are supported.")
        return []

    if orders_df is not None:
        return orders_df.to_dict(orient='records')
    return []
def verify_order(order: dict, order_id:str):
  if order is None:
    return False
  if order.get('order_id') == order_id:
            return True
  return False

def find_order_by_id(orders_list: list, order_id: str):
    """Finds an order in a list of order dictionaries by its order_id."""
    for order in orders_list:
        if order.get('order_id') == order_id:
            return order
    return None

def apply_refund_policy(extracted_data: dict, refund_policy: list, model: str = "openai/gpt-oss-120b", evaluation_date:datetime.date=None):
    """Applies the refund policy to extracted email data using an LLM and validates output."""
    extracted_data_json = json.dumps(extracted_data, indent=2)

    # Ensure evaluation_date is provided, default to today if not
    if evaluation_date is None:
        evaluation_date = datetime.date.today()
    evaluation_date_str = evaluation_date.strftime('%Y-%m-%d')

    user_prompt = f"""
    Given the following extracted email data, refund policy rules, and the current evaluation date,
    determine if a refund is possible. Provide 'YES', 'NO', or 'NOT SURE' for 'refund_possible'
    and a 'reason' explaining the decision.

    Extracted Email Data:
    ```json
    {extracted_data_json}
    ```

    Refund Policy rules:
    {refund_policy}

    Current Evaluation Date: {evaluation_date_str}

    JSON Schema for output:
    ```json
    {{
      "refund_possible": {{ "type": "string", "enum": ["YES", "NO", "NOT SURE"] }},
      "reason": {{ "type": "string" }}
    }}
    ```
    Ensure the output is valid JSON according to this schema.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0.0, # Make it deterministic for policy application
        max_tokens=1024,
        top_p=1.0,
        stream=False,
        response_format={ "type": "json_object" }
    )
    print(f'policy evaluation cost : {response.usage}')

    parsed_content = json.loads(response.choices[0].message.content)

    try:
        # Validate the parsed content against the Pydantic model
        validated_data = RefundPolicyResult.model_validate(parsed_content)
        return validated_data.model_dump() # Return as dict
    except ValidationError as e:
        print(f"[ERROR] Refund policy result validation failed: {e}")
        # In a real application, you might log this error and handle it gracefully
        # For now, we'll return the raw parsed content
        return parsed_content

def send_for_human_approval(data: dict):
    """Simulates sending refund data for human approval."""
    print("\n--- Reviewing the request ---")
    print(json.dumps(data, indent=2))
    # In a real system, this would involve sending an email, API call, etc.

In [45]:
# pip install langgraph
#
# Assumes these already exist in your notebook from Phase 1/2:
#   extract_email_details, load_orders, find_order_by_id,
#   refund_rules_list, apply_refund_policy, send_for_approval
# (send_for_approval must be a PLAIN function here, not a model-callable tool)

import datetime
from typing import TypedDict, Optional, Literal
from langgraph.graph import StateGraph, START, END


# 1. STATE: the shared dictionary every node reads from and writes to
class RefundState(TypedDict, total=False):
    email_body: str
    refund_policy: str
    evaluation_date: Optional[datetime.date]
    extracted: dict
    order: Optional[dict]
    policy_result: dict
    status: str
    reason: str


# 2. NODES: plain functions. Each returns ONLY the keys it wants to update.
def extract_node(state: RefundState) -> dict:            # agentic node (LLM inside)
    return {"extracted": extract_email_details(state["email_body"])}


def lookup_node(state: RefundState) -> dict:             # non-agentic node (pure code)
    orders = load_orders('/content/sample_data/order_data.csv', '.csv')
    order = find_order_by_id(orders, state["extracted"].get("order_number"))
    return {"order": order}


def evaluate_policy_node(state: RefundState) -> dict:    # agentic node (LLM inside)
    rules = refund_rules_list(state["refund_policy"])
    # Judged against the VERIFIED order record, never the email's claims
    result = apply_refund_policy(state["order"], rules,
                                 evaluation_date=state.get("evaluation_date"))
    return {"policy_result": result}


def send_for_approval_node(state: RefundState) -> dict:  # non-agentic node
    send_for_approval({"order": state["order"], "policy_result": state["policy_result"]})
    return {"status": "PENDING_HUMAN_APPROVAL"}


def reject_node(state: RefundState) -> dict:             # non-agentic node
    if state.get("order") is None:
        return {"status": "REJECTED", "reason": "Order not found"}
    # Safely get the reason, providing a default if it's missing
    reason = state["policy_result"].get("reason", "No specific reason provided by policy evaluation (LLM output might be malformed).")
    return {"status": "REJECTED", "reason": reason}


# 3. ROUTERS: plain Python decides the branch. No model involved.
def route_after_lookup(state: RefundState) -> Literal["evaluate_policy", "reject"]:
    return "reject" if state["order"] is None else "evaluate_policy"


def route_after_policy(state: RefundState) -> Literal["send_for_approval", "reject"]:
    return "reject" if state["policy_result"]["refund_possible"] == "NO" else "send_for_approval"


# 4. WIRING: declare the graph
builder = StateGraph(RefundState)
builder.add_node("extract", extract_node)
builder.add_node("lookup", lookup_node)
builder.add_node("evaluate_policy", evaluate_policy_node)
builder.add_node("send_for_approval", send_for_approval_node)
builder.add_node("reject", reject_node)

builder.add_edge(START, "extract")
builder.add_edge("extract", "lookup")                                   # fixed
builder.add_conditional_edges("lookup", route_after_lookup)             # conditional
builder.add_conditional_edges("evaluate_policy", route_after_policy)    # conditional
builder.add_edge("send_for_approval", END)                              # fixed
builder.add_edge("reject", END)

graph = builder.compile()

# 5. VERIFY: print the diagram LangGraph built and compare it to your paper sketch
print(graph.get_graph().draw_mermaid())

# 6. RUN
result = graph.invoke({
    "email_body": "I have an order with id ORDS17807 and it should be refunded as item is not working. "
                  "The amount of order is 30 rupees. The date of order is 20 Sept 2026",
    "refund_policy": "1. The date of order should not be older than 3 days. "
                     "2. Amount of order can not be more than 2500 Rupees.",
    "evaluation_date": datetime.date(2026, 9, 23),
})
print(result["status"], result.get("reason", ""))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract(extract)
	lookup(lookup)
	evaluate_policy(evaluate_policy)
	send_for_approval(send_for_approval)
	reject(reject)
	__end__([<p>__end__</p>]):::last
	__start__ --> extract;
	evaluate_policy -.-> reject;
	evaluate_policy -.-> send_for_approval;
	extract --> lookup;
	lookup -.-> evaluate_policy;
	lookup -.-> reject;
	reject --> __end__;
	send_for_approval --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

email evaluation cost : CompletionUsage(completion_tokens=93, prompt_tokens=304, total_tokens=397, completion_time=0.197159321, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=57), prompt_time=0.103679605, prompt_tokens_details=None, queue_time=0.151704669, total_time=0.300838926)
[
  {
    "rule": "Rule 1",
    "value": "The date of order should not be older than 3 days."
  },
  {
    "rule": "Rule